# 📊 Desacople Crecimiento-Bienestar en América Latina (CEPALSTAT)
## Análisis Empírico Multidimensional: Crecimiento, Pobreza y Salarios Reales (2015–2023)

### 1. Marco Teórico y Contexto de la Investigación
En América Latina y el Caribe, la relación empírica entre el crecimiento del Producto Interno Bruto (PIB) per cápita y la reducción de la pobreza no es lineal ni automática. Choques de precios (como la inflación de alimentos), la alta informalidad laboral y la evolución del **Salario Mínimo Real** determinan si el crecimiento económico se traduce en bienestar efectivo o si deviene en un crecimiento excluyente.

#### Cuadrantes de Desempeño Socioeconómico (Matriz 2x2):
1. **Crecimiento Virtuoso**: $\Delta \text{PIB} > 0$ y $\Delta \text{Pobreza} < 0$. El crecimiento económico reduce efectivamente la pobreza.
2. **Crecimiento Excluyente**: $\Delta \text{PIB} > 0$ y $\Delta \text{Pobreza} \ge 0$. Expansión del producto estéril o concentradora, donde la pobreza se incrementa.
3. **Resiliencia Social**: $\Delta \text{PIB} \le 0$ y $\Delta \text{Pobreza} \le 0$. Contracción del PIB acompañada de mitigación o contención de la pobreza.
4. **Doble Vulnerabilidad**: $\Delta \text{PIB} \le 0$ y $\Delta \text{Pobreza} > 0$. Trampa de involución con recesión y aumento simultáneo de la pobreza.

#### Países Objetivo y Ventanas Temporales:
- **Países**: **Argentina**, **Chile**, **Colombia**, **México**, **Perú**, **Uruguay**.
- **Ventanas de Análisis**:
  - **Pre-Pandemia (2015-2019)**: Dinámica estructural previa al choque sanitario.
  - **Post-Pandemia (2021-2023)**: Reactivación económica heterogénea e impacto inflacionario.

#### Preguntas de Investigación:
- ¿Qué economías lograron situarse en el cuadrante de Crecimiento Virtuoso y cuáles cayeron en Crecimiento Excluyente?
- ¿En qué medida la política de Salario Mínimo Real actuó como mecanismo amortiguador o desacoplador del bienestar frente al PIB?

In [ ]:
# Sección 1: Importación de paquetes y configuración del entorno
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from statscepal import CepalStatClient, DataExtractor
from statscepal.analysis import (
    CountryPeriodMetric,
    DecouplingAnalyzer,
    DecouplingDataset,
    classify_quadrant,
    compute_decoupling_score,
    compute_elasticity,
)

# Configuración de visualización
pd.set_option("display.max_columns", 15)
pd.set_option("display.width", 1000)
print("✅ Entorno inicializado correctamente con statscepal y dependencias analíticas.")

### 2. Extracción y Armonización Automatizada con `DecouplingAnalyzer`

Utilizamos el orquestador `DecouplingAnalyzer` provisto por `statscepal.analysis`. Este orquestador:
1. Extrae de forma paralela las series oficiales de CEPALSTAT:
   - **`3894`**: Tasa de crecimiento anual del PIB real per cápita (%)
   - **`3328`**: Pobreza y pobreza extrema por área geográfica (%)
   - **`340`**: Salario mínimo real (índice 2018=100)
   - **`3289`**: Coeficiente de Gini de desigualdad
2. Aplica las directrices de comparabilidad estadística de CEPAL (delimitando `Total del área urbana` para Argentina y `Nacional` para Chile, Colombia, México, Perú y Uruguay).
3. Realiza la alineación temporal y el cálculo robusto de tasas de crecimiento anualizadas y variaciones en puntos porcentuales.

In [ ]:
# Sección 2: Configuración del cliente y extracción de indicadores CEPALSTAT
client = CepalStatClient()
extractor = DataExtractor(client)
analyzer = DecouplingAnalyzer(extractor)

countries = ["Argentina", "Chile", "Colombia", "México", "Perú", "Uruguay"]
periods = [
    ("2015-2019", 2015, 2019),
    ("2021-2023", 2021, 2023),
]

print(f"📡 Extrayendo y armonizando series para {len(countries)} países en {len(periods)} ventanas temporales...")
dataset = analyzer.analyze_countries(countries=countries, periods=periods)
print(f"✨ Procesamiento completado: {len(dataset.records)} registros temporales generados.")

### 3. Tabla Comparativa de Indicadores y Elasticidades

La **Elasticidad Social del Crecimiento** ($\epsilon_{social}$) cuantifica cuántos puntos porcentuales se reduce (o incrementa) la pobreza por cada 1% de variación del PIB per cápita:
$$\epsilon_{social} = \frac{\Delta \text{Pobreza Total (pp)}}{\Delta \text{PIB per cápita (\%)}}$$

- **$\epsilon < 0$**: Crecimiento pro-pobre o inclusivo (reducción de pobreza ante crecimiento).
- **$\epsilon > 0$**: Desacople social o crecimiento empobrecedor (pobreza aumenta pese a crecimiento positivo).

In [ ]:
# Sección 3: Construcción del DataFrame consolidado de métricas
rows = []
for r in dataset.records:
    rows.append({
        "País": r.country,
        "Periodo": r.period_label,
        "Años": r.years_range,
        "Δ PIB pc (%)": r.gdp_pc_growth_mean,
        "Δ Pob. Tot (pp)": r.poverty_total_change_pp,
        "Δ Pob. Ext (pp)": r.poverty_extreme_change_pp,
        "Δ Sal. Real (%)": r.real_min_wage_change_pct,
        "Δ Gini": r.gini_change,
        "Elasticidad": r.social_elasticity,
        "Cuadrante": r.quadrant,
    })

df_metrics = pd.DataFrame(rows)
df_metrics

In [ ]:
# Tabla comparativa Pre-Pandemia (2015-2019) vs Post-Pandemia (2021-2023)
df_pivot = df_metrics.pivot(
    index="País",
    columns="Periodo",
    values=["Δ PIB pc (%)", "Δ Pob. Tot (pp)", "Δ Sal. Real (%)", "Elasticidad", "Cuadrante"]
)
df_pivot

### 4. Visualización de Cuadrantes Socioeconómicos (Matplotlib)

El plano cartesiano muestra la tasa media de crecimiento anual del PIB per cápita (Eje X) frente a la variación acumulada de la pobreza total en puntos porcentuales (Eje Y).
Esto delimita nítidamente los cuatro cuadrantes de desempeño socioeconómico:
- **Cuadrante I: Crecimiento Virtuoso**
- **Cuadrante II: Crecimiento Excluyente**
- **Cuadrante III: Resiliencia Social**
- **Cuadrante IV: Doble Vulnerabilidad**

In [ ]:
# Sección 4: Gráfico de dispersión de Cuadrantes Socioeconómicos
fig, ax = plt.subplots(figsize=(11, 7.5))

# Líneas de ejes de referencia
ax.axhline(0, color="black", linestyle="--", linewidth=1.1, alpha=0.6)
ax.axvline(0, color="black", linestyle="--", linewidth=1.1, alpha=0.6)

# Estilos por periodo
period_styles = {
    "2015-2019": {"color": "#1f77b4", "marker": "o", "label": "Pre-Pandemia (2015-2019)"},
    "2021-2023": {"color": "#d62728", "marker": "s", "label": "Post-Pandemia (2021-2023)"},
}

for period, style in period_styles.items():
    sub = df_metrics[df_metrics["Periodo"] == period]
    ax.scatter(
        sub["Δ PIB pc (%)"],
        sub["Δ Pob. Tot (pp)"],
        label=style["label"],
        color=style["color"],
        marker=style["marker"],
        s=140,
        edgecolors="black",
        linewidth=0.8,
        alpha=0.9,
        zorder=4,
    )
    for _, row in sub.iterrows():
        if pd.notna(row["Δ PIB pc (%)"]) and pd.notna(row["Δ Pob. Tot (pp)"]):
            offset = (8, 6) if period == "2021-2023" else (8, -12)
            ax.annotate(
                f"{row['País']} ({period})",
                (row["Δ PIB pc (%)"], row["Δ Pob. Tot (pp)"]),
                textcoords="offset points",
                xytext=offset,
                fontsize=9,
                fontweight="semibold",
            )

# Delimitación y etiquetas de los cuadrantes
xlim = ax.get_xlim()
ylim = ax.get_ylim()
span_x = max(abs(xlim[0]), abs(xlim[1]), 5.0)
span_y = max(abs(ylim[0]), abs(ylim[1]), 5.0)
ax.set_xlim(-span_x, span_x)
ax.set_ylim(-span_y, span_y)

ax.text(span_x * 0.45, -span_y * 0.75, "Cuadrante I: Crecimiento Virtuoso\n(PIB ↑, Pobreza ↓)",
        ha="center", va="center", fontsize=10.5, color="darkgreen", fontweight="bold", alpha=0.7,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#e5f5e0", edgecolor="green", alpha=0.4))
ax.text(span_x * 0.45, span_y * 0.75, "Cuadrante II: Crecimiento Excluyente\n(PIB ↑, Pobreza ↑)",
        ha="center", va="center", fontsize=10.5, color="darkred", fontweight="bold", alpha=0.7,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#fee0d2", edgecolor="red", alpha=0.4))
ax.text(-span_x * 0.45, -span_y * 0.75, "Cuadrante III: Resiliencia Social\n(PIB ↓, Pobreza ↓)",
        ha="center", va="center", fontsize=10.5, color="navy", fontweight="bold", alpha=0.7,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#deebf7", edgecolor="blue", alpha=0.4))
ax.text(-span_x * 0.45, span_y * 0.75, "Cuadrante IV: Doble Vulnerabilidad\n(PIB ↓, Pobreza ↑)",
        ha="center", va="center", fontsize=10.5, color="purple", fontweight="bold", alpha=0.7,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#f2f0f7", edgecolor="purple", alpha=0.4))

ax.set_title("Matriz de Desacople Crecimiento-Bienestar en América Latina (CEPALSTAT)", fontsize=13, pad=15, fontweight="bold")
ax.set_xlabel("Δ PIB real per cápita anual promedio (%)", fontsize=11, labelpad=8)
ax.set_ylabel("Δ Pobreza Total acumulada (puntos porcentuales)", fontsize=11, labelpad=8)
ax.legend(loc="upper right", frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
# Variación del Salario Mínimo Real (%) por País y Periodo
fig, ax = plt.subplots(figsize=(10, 5))
pivot_sal = df_metrics.pivot(index="País", columns="Periodo", values="Δ Sal. Real (%)")
pivot_sal.plot(kind="bar", ax=ax, color=["#1f77b4", "#ff7f0e"], width=0.65, edgecolor="black", linewidth=0.6)

ax.axhline(0, color="black", linestyle="--", linewidth=0.8)
ax.set_title("Evolución del Salario Mínimo Real: Variación Acumulada (%)", fontsize=12, pad=12, fontweight="bold")
ax.set_ylabel("Variación Real (%)", fontsize=10)
ax.set_xlabel("País", fontsize=10)
ax.legend(title="Periodo", frameon=True)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 5. Conclusiones y Diagnóstico: El Mecanismo del Salario Mínimo Real

El contraste empírico entre las economías de la región revela hallazgos clave sobre los canales de transmisión macroeconómica hacia el bienestar social:

#### 1. Salario Mínimo Real como Mecanismo de Acople Social (El caso de México):
- **México** implementó una política activa y sostenida de fortalecimiento del **Salario Mínimo Real**, alcanzando un incremento acumulado extraordinario de **+42.8%** en el periodo post-pandemia (2021-2023).
- A pesar de un crecimiento moderado del PIB per cápita, esta recuperación de los ingresos laborales de base permitió una profunda reducción de la pobreza, situando al país de forma sostenida en el cuadrante de **Crecimiento Virtuoso**.

#### 2. Inflación Descontrolada y Destrucción Salarial (El caso de Argentina):
- En contraste, **Argentina** registró tasas positivas de reactivación del PIB per cápita post-pandemia que no se tradujeron en bienestar social. La escalada inflacionaria pulverizó el poder de compra de los salarios más bajos, con una caída del **Salario Mínimo Real** del **-10.7%** (y hasta más del -30% en tramos extendidos).
- Como consecuencia, Argentina cayó en el cuadrante de **Crecimiento Excluyente / Doble Vulnerabilidad**, donde el crecimiento macroeconómico coexiste con un deterioro palpable en los deciles de menores ingresos.

#### 3. Heterogeneidades Estructurales (Chile, Colombia, Perú y Uruguay):
- **Chile y Uruguay** exhibieron trayectorias macro-sociales de mayor estabilidad institucional y menor dispersión en elasticidades sociales.
- **Colombia y Perú** evidenciaron limitaciones asociadas a la alta informalidad del mercado de trabajo, donde el encarecimiento de la canasta básica de alimentos neutralizó parcialmente las ganancias del crecimiento económico.

#### Conclusión de Política Pública:
El crecimiento económico per cápita es indispensable pero insuficiente por sí solo. Sin un mecanismo de transmisión efectivo —liderado por la defensa activa del poder adquisitivo del **Salario Mínimo Real** frente a la inflación de alimentos—, las expansiones del PIB corren el riesgo de convertirse en episodios de crecimiento estéril o excluyente.

In [ ]:
# Resumen ejecutivo para verificación y exportación de bitácora
print("=" * 75)
print("REPORTE DIAGNÓSTICO: DESACOPLE CRECIMIENTO-BIENESTAR (CEPALSTAT)")
print("=" * 75)
for r in dataset.records:
    sal_str = f"{r.real_min_wage_change_pct:+.1f}%" if r.real_min_wage_change_pct is not None else "N/A"
    pob_str = f"{r.poverty_total_change_pp:+.2f} pp" if r.poverty_total_change_pp is not None else "N/A"
    pib_str = f"{r.gdp_pc_growth_mean:+.2f}%"
    print(f"[{r.period_label}] {r.country:<10} | {r.quadrant:<23} | PIB pc: {pib_str:<7} | Pob: {pob_str:<10} | Sal. Real: {sal_str}")
print("=" * 75)